In [0]:
from pyspark.sql.functions import *  
df =  spark.read.table('workspace.bronze.erp_loc_a101')

## Standardize Customer Number

Remove special characters from the customer number to match the key format used in related datasets for joining.

In [0]:

df = df.withColumn('CID' , regexp_replace(col('CID') ,'-' ,'') )


## Standardize Country Values

Convert country codes and names into a consistent format.

In [0]:
df = (
    df.withColumn('CNTRY' , 
                  when(col("CNTRY")  == 'Australia' , 'Australia')
                  .when(col("CNTRY")  == 'Canada' , 'Canada' )
                  .when(col("CNTRY") == 'United Kingdom'  , 'United Kingdom' )
                  .when(col("CNTRY") == 'France' , 'France' )
                  .when(col("CNTRY").isin('Germany' ,'DE')  , 'Germany')
                  .when(col("CNTRY").isin('USA' ,'United States'  , 'US') , 'United States')
                  .otherwise('N/A')
                  
                  )
)

## Rename Columns

Rename columns to follow the Silver layer naming convention.

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "cntry": "country"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Preview Transformed Data

Review the transformed dataset before loading it into the Silver layer.

In [0]:
df.limit(10).display()

## Load to Silver Layer

Write the cleaned and transformed customer location data to the Silver layer.

In [0]:
df.write.mode('overwrite').saveAsTable('workspace.silver.erp_customer_location')

## Verify Data Load

Validate that the customer location data has been successfully loaded into the Silver layer.

In [0]:
%sql
SElECT * 
FROM workspace.silver.erp_customer_location LIMIT 10